In [ ]:
%pip install requests pandas python-dotenv

In [ ]:

import requests

API_KEY = "YOUR_API_KEY"

url = "https://app.ticketmaster.com/discovery/v2/events.json"

params = {
    "apikey": API_KEY,
    "countryCode": "GB",
    "classificationName": "music",
    "size": 20,
    "page": 0
}

response = requests.get(url, params=params)

print(response.status_code)

data = response.json()

print(data)


In [ ]:
events = data["_embedded"]["events"]

print(len(events))
print(events[0])

In [ ]:

import pandas as pd

events = data.get("_embedded", {}).get("events", [])

rows = []

for event in events:

    attractions = event.get("_embedded", {}).get("attractions", [])
    venues = event.get("_embedded", {}).get("venues", [])

    artist = attractions[0] if attractions else {}
    venue = venues[0] if venues else {}

    price_ranges = event.get("priceRanges", [])
    price = price_ranges[0] if price_ranges else {}

    rows.append({
        "event_id": event.get("id"),
        "event_name": event.get("name"),
        "artist": artist.get("name"),
        "artist_id": artist.get("id"),
        "venue": venue.get("name"),
        "city": venue.get("city", {}).get("name"),
        "postcode": venue.get("postalCode"),
        "latitude": venue.get("location", {}).get("latitude"),
        "longitude": venue.get("location", {}).get("longitude"),
        "date": event.get("dates", {}).get(
            "start", {}
        ).get("localDate"),
        "min_price": price.get("min"),
        "max_price": price.get("max"),
        "currency": price.get("currency")
    })

df = pd.DataFrame(rows)

print(df.head())
```

You now have a dataset containing concert information.

Save it:

```python
df.to_csv("uk_concerts.csv", index=False)
```

# 5. Collect thousands of concerts

The API uses pagination. You can request multiple pages rather than limiting yourself to the first 20 events.

Here's a function to collect multiple pages.

<CodeBlock language="python" editable>
import requests
import pandas as pd
import time

API_KEY = "YOUR_API_KEY"

def get_concerts(total_pages=10):

    all_events = []

    url = "https://app.ticketmaster.com/discovery/v2/events.json"

    for page in range(total_pages):

        params = {
            "apikey": API_KEY,
            "countryCode": "GB",
            "classificationName": "music",
            "size": 100,
            "page": page
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        data = response.json()

        events = data.get("_embedded", {}).get("events", [])

        if not events:
            break

        all_events.extend(events)

        print(f"Page {page}: collected {len(events)} events")

        time.sleep(0.25)

    return all_events


events = get_concerts(total_pages=10)

print("Total events:", len(events))


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("TICKETMASTER_API_KEY")